# LLM Fine-Tuning & Alignment (LoRA & DPO)
**Level:** Professional | **Interactive Machine Learning Curriculum**

> Supervised Fine-Tuning (SFT), Parameter-Efficient Fine-Tuning (PEFT: LoRA, QLoRA), and Alignment (RLHF vs DPO).

---

### 📖 Conceptual Foundations

#### The Post-Pretraining Journey

Raw base foundation models (e.g., Llama-3-8B-Base) are trained purely for next-token continuation over trillions of web tokens. Turning them into reliable assistants involves two critical post-training phases:
1. **Instruction Tuning / SFT (Supervised Fine-Tuning):** Teaches the model conversational interaction patterns, formatting compliance, and tool-use syntax.
2. **Preference Alignment (RLHF / DPO):** Aligns outputs with human values of **Helpfulness, Honesty, and Harmlessness (HHH)**.


#### LoRA: Low-Rank Adaptation (Hu et al., Microsoft 2021)

Full fine-tuning updates the entire parameter matrix $W_0 \in \mathbb{R}^{d \times k}$. **LoRA** hypothesizes that weight changes $\Delta W$ have a low **intrinsic rank** $r \ll \min(d, k)$. It freezes $W_0$ and decomposes updates into two low-rank matrices:

$$W = W_0 + \Delta W = W_0 + \frac{\alpha}{r} (B \cdot A)$$

Where $A \in \mathbb{R}^{r \times k}$ is initialized with Gaussian noise $\mathcal{N}(0, \sigma^2)$, $B \in \mathbb{R}^{d \times r}$ is initialized to zero, and $\alpha$ is a constant scaling hyperparameter.

- **Zero Inference Latency:** During deployment, $B \cdot A$ is mathematically added directly to $W_0$, introducing zero latency overhead!
- **VRAM Savings:** For $r=8$ or $16$, trainable parameters drop by **$99.9\%$**, slashing optimizer VRAM requirements.


#### Evolution of LLM Alignment Paradigms

| Alignment Paradigm | Methodology & Architecture | Pros | Cons & Challenges |
| --- | --- | --- | --- |
| RLHF (PPO) | Train a Reward Model $R(x, y)$ on human pairwise rankings; optimize policy $\pi_\theta$ via Proximal Policy Optimization (PPO) with a KL divergence penalty to reference model $\pi_{ref}$. | Gold standard for nuanced human preference; powerful exploration. | Extremely complex pipeline: requires maintaining 4 models concurrently in VRAM (Actor, Critic, Reference, Reward). |
| Direct Preference Optimization (DPO) | Mathematically proves that the constrained RL objective can be reparameterized to express the reward directly in terms of policy probability ratios $\frac{\pi_\theta(y|x)}{\pi_{ref}(y|x)}$. | Requires NO reward model and NO reinforcement learning loop; stable supervised loss. | Sensitive to noisy preference labels; susceptible to likelihood displacement on out-of-distribution prompts. |
| Kahneman-Tversky Optimization (KTO) | Aligns models directly on binary signals (thumbs up / thumbs down) using Prospect Theory utility functions. | Eliminates the need for paired $(y_{win}, y_{lose})$ data; works on cheaper unpaired feedback. | Slightly less sample efficient on subtle ranking nuances. |


#### QLoRA: Quantized Low-Rank Adaptation (Dettmers et al., 2023)

**QLoRA** enables fine-tuning a 70B parameter model on a single 48GB GPU by combining:
1. **4-bit NormalFloat (NF4):** Information-theoretically optimal quantile quantization for normally distributed weights.
2. **Double Quantization (DQ):** Quantizes the quantization constants themselves, saving $0.37$ bits per parameter.
3. **Paged Optimizers:** Uses CUDA Unified Memory to automatically page optimizer states to CPU RAM during memory spikes.



In [ ]:
# %matplotlib inline
import numpy as np

class LoRALinear:
    def __init__(self, in_features, out_features, rank=4, lora_alpha=16):
        self.in_features = in_features
        self.out_features = out_features
        self.rank = rank
        self.scaling = lora_alpha / rank
        
        # Frozen base weights (simulating 8B parameter weights)
        self.W0 = np.random.randn(out_features, in_features) * 0.02
        self.W0_frozen = True
        
        # Trainable low-rank adapter matrices
        self.A = np.random.randn(rank, in_features) * 0.01  # Gaussian init
        self.B = np.zeros((out_features, rank))             # Zero init so delta starts at 0
        
    def forward(self, x):
        # Base output + scaled low-rank output
        base_out = np.dot(x, self.W0.T)
        lora_out = np.dot(np.dot(x, self.A.T), self.B.T) * self.scaling
        return base_out + lora_out
        
    def merge_weights_for_inference(self):
        # Permanently fold delta into W0: Zero inference latency overhead!
        self.W0 += (np.dot(self.B, self.A) * self.scaling)
        self.B = None
        self.A = None
        print('LoRA successfully merged into base weights!')

# Instantiate layer: 4096 -> 4096 with rank r=8
layer = LoRALinear(in_features=4096, out_features=4096, rank=8, lora_alpha=16)
x = np.random.randn(2, 4096)
y_before = layer.forward(x)

# Mock training step updating B
layer.B += np.random.randn(4096, 8) * 0.05
y_after = layer.forward(x)
print(f'Difference in output after adapter update: {np.linalg.norm(y_after - y_before):.4f}')

# Merge for production serving
layer.merge_weights_for_inference()
y_merged = np.dot(x, layer.W0.T)
print(f'Max difference between merged W0 and adapter forward pass: {np.max(np.abs(y_merged - y_after)):.2e}')


---

### 🎯 Hands-On Experiments & Challenges
1. **Experiment with parameters:** Modify the hyperparameters or dataset sizes in the cell above.
2. **Add visual diagnostics:** Plot loss curves or feature distributions using Matplotlib/Seaborn.
3. **Test with new data:** Try running the model on synthetic noisy data or custom inputs.


In [ ]:
# Your custom experiment code here:

